# Lab 02: Supervised ML & PROC LOGISTIC Bridge

**WatSPEED Agentic AI Prep Hub | Stage 2 of 4**

---

## What Are We Doing and Why?

In SAS you'd run `PROC LOGISTIC` to model a binary outcome like `High_AI_Trust` (1/0). In Python/scikit-learn the workflow is nearly identical — but it gives you a **test set accuracy** and **ROC-AUC** that SAS's fit statistics don't naturally produce.

| SAS | Python/sklearn |
|---|---|
| `PROC LOGISTIC DATA=clean;` | `LogisticRegression().fit(X_train, y_train)` |
| `MODEL High_AI_Trust = Age_Group Education;` | `X = df[['age_enc', 'edu_enc']]` |
| `WEIGHT survey_weight;` | `sample_weight=weights` param |
| Fit statistics (AIC, -2LogL) | `accuracy_score`, `roc_auc_score` |
| `OUTPUT OUT=preds P=pred_prob;` | `clf.predict_proba(X_test)[:, 1]` |

## ✏️ Your Tasks
1. Run all cells and read the output
2. **Change `test_size=0.20` to `0.30`** — does accuracy change?
3. **Change `C=1.0` to `C=0.1`** (weaker regularization) — what happens to AUC?
4. **Add `Education_Level` as a predictor** by including `edu_enc` in `feature_cols`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder

print('All imports successful!')

In [ ]:
# ============================================================
# Cell 2: Dataset — AI Trust Insights (n=1200, sample shown)
# ============================================================
np.random.seed(42)
n = 200

age_groups  = np.random.choice(['18-29','30-44','45-59','60+'], n, p=[0.25,0.35,0.25,0.15])
edu_levels  = np.random.choice(['High School','Bachelor\'s','Master\'s','PhD'], n, p=[0.30,0.35,0.25,0.10])
ai_risk     = np.random.randint(1, 6, n)           # Likert 1-5
tech_fam    = np.random.randint(1, 6, n)           # Technology familiarity 1-5
weights     = np.random.uniform(0.75, 1.75, n)     # Survey weights

# Outcome: High AI Trust (1/0) — inversely related to ai_risk
logit = 1.5 - 0.5 * ai_risk + 0.3 * tech_fam
prob  = 1 / (1 + np.exp(-logit))
ai_trust = (np.random.random(n) < prob).astype(int)

df = pd.DataFrame({
    'age_group': age_groups,
    'education': edu_levels,
    'ai_risk':   ai_risk,
    'tech_fam':  tech_fam,
    'weight':    weights,
    'high_ai_trust': ai_trust
})

print(f'Dataset: {len(df)} respondents')
print(df.head())
print(f'\nOutcome distribution:')
print(df['high_ai_trust'].value_counts())

### The "Why": Why do we encode categorical variables manually?

> **The Legacy Friction:** In SAS, you simply add a `CLASS Age_Group;` statement to your `PROC LOGISTIC` and SAS handles the dummy coding behind the scenes. This is convenient for statistics, but dangerous for production deployment because if a new category appears later, the model breaks unexpectedly.
>
> **The Python/RAP Approach:** `scikit-learn` forces you to convert text into numbers explicitly before modeling. While it feels like an extra step, it forces you to build an explicit **Encoding Pipeline**. When you deploy this model as an API or Agent tool later, that pipeline guarantees that incoming strings are mapped to the exact same integers used during training, preventing silent production crashes.


In [ ]:
# ============================================================
# Cell 3: Encode Categorical Variables
# SAS: PROC LOGISTIC uses CLASS statement for automatic dummy coding
# Python: We encode manually (or TabFM skips this step entirely!)
# ============================================================
le_age = LabelEncoder()
le_edu = LabelEncoder()

df['age_enc'] = le_age.fit_transform(df['age_group'])
df['edu_enc'] = le_edu.fit_transform(df['education'])

print('Age group encoding:')
for orig, enc in zip(le_age.classes_, range(len(le_age.classes_))):
    print(f'  {orig:10s} -> {enc}')

print('\nEducation encoding:')
for orig, enc in zip(le_edu.classes_, range(len(le_edu.classes_))):
    print(f'  {orig:15s} -> {enc}')

### The "Why": Why do we split our data (Train vs Test)?

> **The Legacy Friction:** In traditional sociological or econometric SAS workflows, you typically run `PROC LOGISTIC` on the *entire* dataset. Your goal is inference (understanding p-values and coefficients). 
>
> **The Python/RAP Approach:** In modern machine learning (and Agentic AI workflows), your goal is **prediction on unseen data**. If you evaluate a model on the same data it learned from, you will vastly overestimate its accuracy (Overfitting). The `train_test_split` is mandatory. We hide 20% of the data from the model during training, and use it at the very end to prove the model actually works.


In [ ]:
# ============================================================
# Cell 4: Train/Test Split & Logistic Regression
# SAS equivalent: no built-in train/test split — you'd use
#   PROC SURVEYSELECT to create a holdout sample manually
# ============================================================

# ✏️ EDIT THESE PARAMETERS
feature_cols = ['ai_risk', 'tech_fam', 'age_enc']  # try adding 'edu_enc'
test_size    = 0.20   # 20% held out for testing; try 0.30
C_param      = 1.0    # Regularization strength; try 0.1 or 10.0

X = df[feature_cols].values
y = df['high_ai_trust'].values
w = df['weight'].values

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=test_size, random_state=42, stratify=y
)

clf = LogisticRegression(C=C_param, max_iter=500)
clf.fit(X_train, y_train, sample_weight=w_train)  # uses WEIGHT statement like SAS

y_pred  = clf.predict(X_test)
y_prob  = clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print('=== LOGISTIC REGRESSION RESULTS ===')
print(f'Features used: {feature_cols}')
print(f'Train size: {len(X_train)}  |  Test size: {len(X_test)}')
print(f'Test Accuracy:  {acc*100:.1f}%')
print(f'ROC-AUC Score:  {auc:.3f}')
print()
print('SAS PROC LOGISTIC equivalent: Model converged successfully')
print()
print(classification_report(y_test, y_pred, target_names=['Low Trust','High Trust']))

In [ ]:
# ============================================================
# Cell 5: Coefficient Interpretation (like SAS PROC LOGISTIC output)
# In SAS: you get Odds Ratios in the output table automatically
# ============================================================
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'log_odds': clf.coef_[0],
    'odds_ratio': np.exp(clf.coef_[0])
})

print('=== COEFFICIENT TABLE (SAS Odds Ratio Estimates equivalent) ===')
print(coef_df.to_string(index=False))
print(f'\nIntercept (log-odds): {clf.intercept_[0]:.4f}')
print('\nInterpretation: Odds Ratio < 1 means NEGATIVE association with High AI Trust')